# R7 — Log, register, version, serve

The claims agent is treated as a product: logged as an MLflow pyfunc model, registered in Unity Catalog as claims.genai.claims_agent, versioned and served. Two versions differ only in their system prompt (agent/prompts/system_v1.md and system_v2.md); each version is tagged system_prompt_sha - the first 12 hex characters of the SHA-256 of its prompt file text, leading and trailing whitespace removed - and the champion alias marks the v2 version that serves live traffic on the CPU endpoint claims-agent. Evidence: evidence/r7_registry.json.

In [0]:
# Config - catalog, models, paths, warehouse, registry name and endpoint specs
import os

CATALOG = "claims"
SCHEMA = "genai"
CHAT_MODEL = "databricks-meta-llama-3-3-70b-instruct"
EMBED_MODEL = "databricks-gte-large-en"
REPO_DIR = "/Workspace/Users/deepaksinghal@cyntexa.com/-DE---Scenario-based-Assignment-1/Gen AI - Paper 5 - Scenario Based Assignment"
R1_TOOLS = ["lookup_claim", "plan_at", "search_policy", "compute_payable"]

REGISTERED_MODEL = f"{CATALOG}.{SCHEMA}.claims_agent"   # UC registered model
ENDPOINT_NAME = "claims-agent"                          # CPU serving endpoint (R7 + R8 app resource)
WAREHOUSE_ID = "c5bf9dda5ab27d2f"                       # Serverless Starter Warehouse - tools run through SQL statements

spark.sql(f"USE CATALOG {CATALOG}")

import hashlib
import requests
from databricks.sdk import WorkspaceClient

w = WorkspaceClient()
AUTH = w.config.authenticate()["Authorization"]          # Bearer token, never printed or saved
HOST = w.config.host if w.config.host.startswith("http") else "https://" + w.config.host

print(f"registered model: {REGISTERED_MODEL}  endpoint: {ENDPOINT_NAME}")
print(f"chat model: {CHAT_MODEL}  warehouse: {WAREHOUSE_ID}")

# probe claim: deterministic first Paper 4 probe claim (same probe used in R6)
PROBE_CLAIM = spark.sql(
    "SELECT claim_id FROM claims.genai.adjudication_probes ORDER BY claim_id LIMIT 1"
).collect()[0].claim_id
print(f"probe claim: {PROBE_CLAIM}")

In [0]:
# The v2 system prompt - a production revision of system_v1.md: same tool contract, plus
# tone/brevity rules, a flag for fields the tools did not return, and a verification footer.
import hashlib

V2_PROMPT = """You are the Saffron Shield claims assistant serving insurance adjusters in production.
Answer only questions about Saffron Shield claims, plans and policy documents; anything else gets a brief refusal with no extra detail.
For claim facts, status or amounts, always call lookup_claim first, and plan_at whenever plan terms matter; repeat the tool output verbatim instead of relying on memory.
For policy wording questions, call search_policy, answer strictly from the returned sections, and cite each one by its chunk id like 03_Waiting_Periods_Group_Plans.md#pre-existing-disease-waiting-period.
Never state or estimate a payout yourself: payouts come only from the compute_payable tool, and you confirm the payable amount matches the tool result before replying.
Keep replies short, factual and adjuster-friendly, flag any field the tools did not return, and end with Verified against claims tools. when a claim lookup supported the answer."""

v1_path = os.path.join(REPO_DIR, "agent", "prompts", "system_v1.md")
v2_path = os.path.join(REPO_DIR, "agent", "prompts", "system_v2.md")
with open(v1_path) as f:
    V1_PROMPT = f.read()
with open(v2_path, "w") as f:
    f.write(V2_PROMPT)
with open(v2_path) as f:  # round-trip
    assert f.read() == V2_PROMPT

# fingerprint: first 12 hex chars of SHA-256 of the prompt text, leading/trailing whitespace removed
SHA_V1 = hashlib.sha256(V1_PROMPT.strip().encode()).hexdigest()[:12]
SHA_V2 = hashlib.sha256(V2_PROMPT.strip().encode()).hexdigest()[:12]

print(f"v1: {len(V1_PROMPT.split())} words  system_prompt_sha={SHA_V1}")
print(f"v2: {len(V2_PROMPT.split())} words  system_prompt_sha={SHA_V2}")
assert len(V2_PROMPT.split()) >= 40, "v2 prompt must be at least 40 words"
assert SHA_V1 != SHA_V2, "prompts must differ"
print("wrote", v2_path)

In [0]:
# The servable agent: a pyfunc model that runs the R2 tool loop at serving time.
# Credentials are NOT written into the model - at serving time Databricks injects
# DATABRICKS_HOST / DATABRICKS_TOKEN for the resources declared at log time (chat model
# serving endpoint, SQL warehouse, the four UC functions); locally we set them from the
# notebook session. Tools run through the SQL Statement Execution API against the same
# claims.genai functions; the LLM is the temperature-0 chat model.
import json
import mlflow


class ClaimsAgentModel(mlflow.pyfunc.PythonModel):
    """Tool-calling claims agent; the only per-version input is the system prompt."""

    def __init__(self, system_prompt: str, chat_endpoint: str, warehouse_id: str):
        self.system_prompt = system_prompt
        self.chat_endpoint = chat_endpoint
        self.warehouse_id = warehouse_id
        self.max_steps = 4

    # ---- credentials / transport -------------------------------------------------
    # Resolve auth through the Databricks SDK's standard chain so the model picks up
    # whatever the serving container injects (DATABRICKS_TOKEN, or OAuth M2M vars).
    def _wc(self):
        if getattr(self, "_wclient", None) is None:
            try:
                from databricks.sdk import WorkspaceClient
                self._wclient = WorkspaceClient()
            except Exception as e:  # noqa: BLE001 - surface what was actually injected
                keys = sorted(k for k in os.environ
                              if k.startswith(("DATABRICKS", "MLFLOW")))
                raise RuntimeError(
                    f"serving credentials not injected ({str(e)[:200]}); env keys: {keys}") from e
        return self._wclient

    def _host(self):
        return self._wc().config.host.rstrip("/")

    def _auth(self):
        return {"Authorization": self._wc().config.authenticate()["Authorization"],
                "Content-Type": "application/json"}

    def _chat(self, messages, tools):
        body = {"messages": messages, "tools": tools, "temperature": 0, "max_tokens": 1024}
        r = requests.post(f"{self._host()}/serving-endpoints/{self.chat_endpoint}/invocations",
                          headers=self._auth(), json=body, timeout=120)
        r.raise_for_status()
        return r.json()

    def _sql(self, statement):
        r = requests.post(f"{self._host()}/api/2.0/sql/statements",
                          headers=self._auth(), timeout=120,
                          json={"warehouse_id": self.warehouse_id, "statement": statement,
                                "wait_timeout": "30s", "format": "JSON_ARRAY",
                                "disposition": "INLINE"})
        r.raise_for_status()
        j = r.json()
        st = j.get("status", {})
        if st.get("state") != "SUCCEEDED":
            raise RuntimeError(st.get("error", {}).get("message", st.get("state")))
        cols = [c["name"] for c in j["manifest"]["schema"]["columns"]]
        return [dict(zip(cols, row)) for row in j["result"].get("data_array", [])]

    # ---- tools -------------------------------------------------------------------
    @staticmethod
    def _lit(v):
        if v is None:
            return "NULL"
        if isinstance(v, bool):
            return "TRUE" if v else "FALSE"
        if isinstance(v, (int, float)):
            return repr(v)
        return "'" + str(v).replace("'", "''") + "'"

    def _run_tool(self, name, args):
        L = self._lit
        if name == "lookup_claim":
            sql = f"SELECT * FROM claims.genai.lookup_claim({L(args['claim_id'])})"
        elif name == "plan_at":
            sql = f"SELECT * FROM claims.genai.plan_at({L(args['member_id'])}, DATE {L(args['as_of_date'])})"
        elif name == "search_policy":
            sql = f"SELECT * FROM claims.genai.search_policy({L(args['query'])})"
        elif name == "compute_payable":
            sql = (f"SELECT claims.genai.compute_payable({L(args['plan_code'])}, {L(args['is_pre_existing'])}, "
                   f"{L(args['policy_tenure_months'])}, {L(args['days_since_policy_start'])}, "
                   f"{L(args['is_network_at_admission'])}, {L(args['room_rent_per_day'])}, "
                   f"{L(args['billed_amount'])}) AS output")
        else:
            raise ValueError(f"unknown tool {name}")
        rows = self._sql(sql)
        if name == "compute_payable" and rows:
            try:
                return json.loads(rows[0]["output"])
            except (json.JSONDecodeError, KeyError):
                return rows
        return rows

    TOOL_DEFS = [
        {"type": "function", "function": {
            "name": "lookup_claim",
            "description": "Returns a claim's stored facts by claim_id: status, dates, billed/approved amounts, plan at admission, pre-existing flag, tenure, network flag, room rent.",
            "parameters": {"type": "object", "properties": {"claim_id": {"type": "string"}}, "required": ["claim_id"]}}},
        {"type": "function", "function": {
            "name": "plan_at",
            "description": "Returns the member's plan version and terms in force on a date (use the claim's admission date).",
            "parameters": {"type": "object", "properties": {"member_id": {"type": "string"}, "as_of_date": {"type": "string"}}, "required": ["member_id", "as_of_date"]}}},
        {"type": "function", "function": {
            "name": "search_policy",
            "description": "Hybrid search over the policy documents; returns the top 5 policy chunk ids, best first. Cite answers by chunk id.",
            "parameters": {"type": "object", "properties": {"query": {"type": "string"}}, "required": ["query"]}}},
        {"type": "function", "function": {
            "name": "compute_payable",
            "description": "Computes the adjudication decision (payable, waiting period, network, room-rent deduction, copay, reason code, payable amount) from the claim facts. The only source of payout figures.",
            "parameters": {"type": "object", "properties": {
                "plan_code": {"type": "string"}, "is_pre_existing": {"type": "boolean"},
                "policy_tenure_months": {"type": "integer"}, "days_since_policy_start": {"type": "integer"},
                "is_network_at_admission": {"type": "boolean"}, "room_rent_per_day": {"type": "integer"},
                "billed_amount": {"type": "number"}},
                "required": ["plan_code", "is_pre_existing", "policy_tenure_months", "days_since_policy_start",
                             "is_network_at_admission", "room_rent_per_day", "billed_amount"]}}},
    ]

    # ---- the loop ----------------------------------------------------------------
    def _run(self, question: str):
        messages = [{"role": "system", "content": self.system_prompt},
                    {"role": "user", "content": question}]
        calls = []
        try:  # identity the serving container runs SQL as (for governance diagnostics)
            server_user = self._sql("SELECT current_user() AS u")[0]["u"]
        except Exception as e:  # noqa: BLE001
            server_user = f"unresolved: {e}"
        for _ in range(self.max_steps):
            msg = self._chat(messages, self.TOOL_DEFS)["choices"][0]["message"]
            tcs = msg.get("tool_calls") or []
            if not tcs:
                return {"question": question, "answer": (msg.get("content") or "").strip(),
                        "tool_calls": calls, "server_user": server_user}
            messages.append(msg)
            for tc in tcs:
                name = tc["function"]["name"]
                args = json.loads(tc["function"]["arguments"] or "{}")
                try:
                    result = self._run_tool(name, args)
                except Exception as e:  # noqa: BLE001 - hand the error back to the model
                    result = {"tool_error": str(e)}
                calls.append({"name": name, "args": args})
                messages.append({"role": "tool", "tool_call_id": tc["id"],
                                 "content": json.dumps(result, default=str)})
        return {"question": question, "answer": "", "tool_calls": calls,
                "server_user": server_user}

    def predict(self, context, model_input, params=None):
        if isinstance(model_input, str):
            questions = [model_input]
        elif isinstance(model_input, dict):
            q = model_input.get("question")
            questions = list(q) if isinstance(q, list) else [q]
        else:
            questions = list(model_input["question"])
        out = [self._run(str(q)) for q in questions]
        return out[0] if len(out) == 1 else out


print("ClaimsAgentModel defined")

In [0]:
# Log both versions (same code, different prompt) with signature + input example + the
# resources the served container needs, then run the isolated uv validation on the
# champion artifact - it must load and predict from a fresh environment before registration.
import pandas as pd
from mlflow.models import infer_signature
from mlflow.models.resources import (DatabricksFunction, DatabricksSQLWarehouse,
                                     DatabricksServingEndpoint, DatabricksTable,
                                     DatabricksVectorSearchIndex)

# UC functions run with caller's privileges: the provisioned principal also needs the
# underlying tables and the vector-search index the tool functions read.
AGENT_TABLES = ["claims.gold.claims_enriched", "claims.silver.policy_history",
                "claims.raw.plans", "claims.genai.policy_chunks"]
RESOURCES = [DatabricksServingEndpoint(endpoint_name=CHAT_MODEL),
             DatabricksSQLWarehouse(warehouse_id=WAREHOUSE_ID)] + \
            [DatabricksFunction(function_name=f"claims.genai.{t}") for t in R1_TOOLS] + \
            [DatabricksTable(table_name=t) for t in AGENT_TABLES] + \
            [DatabricksVectorSearchIndex(index_name="claims.genai.policy_chunks_idx")]

EXAMPLE_IN = {"question": "What's the status of claim CLM-10039?"}
EXAMPLE_OUT = {"question": EXAMPLE_IN["question"], "answer": "settled", "tool_calls": []}
SIGNATURE = infer_signature(EXAMPLE_IN, EXAMPLE_OUT)
PIP = [f"mlflow=={mlflow.__version__}", "requests", "pandas", "databricks-sdk"]

# the validation subprocess needs the same credentials the serving container gets injected
os.environ.setdefault("DATABRICKS_HOST", HOST)
_tok = AUTH.split(" ", 1)[1]
os.environ.setdefault("DATABRICKS_TOKEN", _tok)

# log inside an active MLflow run: the UC model version then carries a source run, which is
# what Databricks uses to provision the injected serving credentials for declared resources
# (without it the served container has no DATABRICKS_HOST/DATABRICKS_TOKEN).
def log_version(prompt: str, run_name: str):
    with mlflow.start_run(run_name=run_name) as run:
        return mlflow.pyfunc.log_model(
            name="claims_agent_model",
            python_model=ClaimsAgentModel(prompt, CHAT_MODEL, WAREHOUSE_ID),
            signature=SIGNATURE,
            input_example=EXAMPLE_IN,
            resources=RESOURCES,
            pip_requirements=PIP,
        )

# re-log only when the model code changed (stamp bump forces a fresh log)
_STAMP = "claims-agent-model-2026-11-16e-identity"
if globals().get("_LOGGED_STAMP") != _STAMP:
    mi_v1 = log_version(V1_PROMPT, "claims_agent_v1")
    mi_v2 = log_version(V2_PROMPT, "claims_agent_v2")
    _LOGGED_STAMP = _STAMP
print("v1 artifact:", mi_v1.model_uri, " run:", mi_v1.run_id)
print("v2 artifact:", mi_v2.model_uri, " run:", mi_v2.run_id)

# isolated check (fresh uv env installs the declared requirements from scratch).
# On failure, reproduce the same isolated run with captured stderr so the root cause is visible.
import json, shutil, subprocess, sys, tempfile


def uv_isolated_check():
    mlflow.models.predict(model_uri=mi_v2.model_uri, input_data=EXAMPLE_IN, env_manager="uv")
    print("isolated uv validation passed for the champion artifact")


try:
    uv_isolated_check()
except Exception as e:  # noqa: BLE001
    print("mlflow uv check failed:", str(e)[:400])
    model_dir = mlflow.artifacts.download_artifacts(artifact_uri=mi_v2.model_uri)
    if not shutil.which("uv"):
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    venv = tempfile.mkdtemp(prefix="uvcheck_")
    py = os.path.join(venv, "bin", "python")
    def run(cmd, **kw):
        r = subprocess.run(cmd, capture_output=True, text=True, timeout=1200, **kw)
        print(f"$ {' '.join(cmd[:3])}... rc={r.returncode}")
        if r.returncode != 0:
            print(r.stdout[-1500:])
            print(r.stderr[-1500:])
        return r
    if run(["uv", "venv", venv, "--python", sys.executable]).returncode == 0 and \
       run(["uv", "pip", "install", "--python", py, "-r",
            os.path.join(model_dir, "requirements.txt")]).returncode == 0:
        input_path = os.path.join(model_dir, "_validation_input.json")
        with open(input_path, "w") as f:
            json.dump({"dataframe_records": [{"question": EXAMPLE_IN["question"]}]}, f)
        backend = os.path.join(os.path.dirname(mlflow.__file__), "pyfunc",
                               "_mlflow_pyfunc_backend_predict.py")
        r = run([py, backend, "--model-uri", f"file://{model_dir}",
                 "--content-type", "json", "--input-path", input_path],
                env={**os.environ})
        if r.returncode == 0:
            print("manual uv reproduction PASSED:", r.stdout[-500:])
        else:
            raise RuntimeError("isolated uv validation failed - see stderr above")
    else:
        raise RuntimeError("uv environment/install failed - see output above")

In [0]:
# Register in Unity Catalog: v1 carries the v1 prompt's sha, v2 the v2's, champion alias on
# v2 (the version that serves live traffic). If UC registration is unavailable, fall back to
# logging tagged runs in the experiment (mode=experiment).
mlflow.set_registry_uri("databricks-uc")

MODE = "uc"
VERSIONS = {}
try:
    from mlflow import MlflowClient
    reg = MlflowClient(registry_uri="databricks-uc")
    for tag, sha, mi in ((1, SHA_V1, mi_v1), (2, SHA_V2, mi_v2)):
        mv = mlflow.register_model(model_uri=mi.model_uri, name=REGISTERED_MODEL,
                                   await_registration_for=600)
        reg.set_model_version_tag(REGISTERED_MODEL, mv.version, "system_prompt_sha", sha)
        reg.set_model_version_tag(REGISTERED_MODEL, mv.version, "agent_version", str(tag))
        VERSIONS[tag] = {"version": mv.version, "system_prompt_sha": sha}
    reg.set_registered_model_alias(name=REGISTERED_MODEL, alias="champion",
                                   version=VERSIONS[2]["version"])
    VERSIONS[2]["alias"] = "champion"
    resolved = reg.get_model_version_by_alias(name=REGISTERED_MODEL, alias="champion")
    assert resolved.version == VERSIONS[2]["version"]
except Exception as e:  # noqa: BLE001 - documented fallback path
    print(f"UC registration unavailable ({e}); falling back to experiment mode")
    MODE = "experiment"
    exp = mlflow.get_experiment_by_name("claims_agent_registry") or \
          mlflow.create_experiment("claims_agent_registry")
    mlflow.set_experiment(experiment_id=exp)
    for tag, sha, mi in ((1, SHA_V1, mi_v1), (2, SHA_V2, mi_v2)):
        with mlflow.start_run(run_name=f"claims_agent_v{tag}") as run:
            mlflow.log_artifact(os.path.join(REPO_DIR, "agent", "prompts",
                                             f"system_v{tag}.md"), artifact_path="prompts")
            mlflow.set_tags({"agent_version": str(tag), "system_prompt_sha": sha,
                             "alias": "champion" if tag == 2 else "",
                             "served_from": "app" if tag == 2 else ""})
            VERSIONS[tag] = {"run_id": run.info.run_id, "system_prompt_sha": sha,
                             "alias": "champion" if tag == 2 else None}

print(f"mode={MODE}  versions={VERSIONS}")

In [0]:
# Registry hygiene (user-approved): delete ONLY the two superseded versions 5 and 6 of
# claims.genai.claims_agent - duplicates of v7/v8 (same prompts, same shas) registered
# before the table/index resources were added. v7 (v1 prompt) and v8 (champion) are kept.
keep = {str(VERSIONS[1]["version"]), str(VERSIONS[2]["version"])}
superseded = [v for v in ("1", "2", "3", "4", "5", "6","7","8") if v not in keep]
print("deleting superseded versions:", superseded)
for v in superseded:
    try:
        reg.delete_model_version(REGISTERED_MODEL, v)
        print(f"deleted superseded version {v}")
    except Exception as de:  # noqa: BLE001 - already gone
        print(f"version {v}:", str(de)[:80])

print("live versions:", [(v, reg.get_model_version(REGISTERED_MODEL, v).tags["system_prompt_sha"],
                          reg.get_model_version(REGISTERED_MODEL, v).aliases)
                         for v in sorted(mv.version for mv in
                                         reg.search_model_versions(f"name='{REGISTERED_MODEL}'"))])

In [0]:
# Serve the champion on the CPU endpoint claims-agent (returns immediately; readiness is
# polled in the next cell). CPU is required by the paper; scale-to-zero keeps it cheap.
from databricks.sdk.errors import NotFound
from databricks.sdk.service.serving import (EndpointCoreConfigInput, ServedEntityInput,
                                            ServingModelWorkloadType)

champion_version = VERSIONS[2]["version"]

workload_response = w.api_client.do("GET", "/api/2.0/serving-endpoints:workload-configs")
supported = {c["workload_type"]: [s["key"] for s in c.get("workload_sizes", [])]
             for c in workload_response.get("workload_configs", [])}
assert "CPU" in supported, f"no CPU workload available: {supported}"
size = "Small" if "Small" in supported["CPU"] else supported["CPU"][0]

served_entity = ServedEntityInput(
    entity_name=REGISTERED_MODEL,
    entity_version=str(champion_version),
    workload_type=ServingModelWorkloadType("CPU"),
    workload_size=size,
    scale_to_zero_enabled=True,
)

try:
    existing = w.serving_endpoints.get(ENDPOINT_NAME)
    cur = [e.entity_name for e in (existing.config.served_entities or [])]
    if cur != [REGISTERED_MODEL]:
        raise ValueError(f"endpoint {ENDPOINT_NAME} serves {cur}; refusing to repurpose")
    w.serving_endpoints.update_config(name=ENDPOINT_NAME, served_entities=[served_entity])
    print("updating existing endpoint")
except NotFound:
    w.serving_endpoints.create(name=ENDPOINT_NAME,
                               config=EndpointCoreConfigInput(name=ENDPOINT_NAME,
                                                              served_entities=[served_entity]))
    print("created endpoint")

print(f"claims-agent -> {REGISTERED_MODEL} v{champion_version} (champion), CPU/{size}, scale-to-zero")

In [0]:
# Wait for the endpoint to be READY (typically ~10-15 minutes), then smoke-test it the way
# the R8 chat page will: a question over the serving endpoint's invocations API.
import time
from databricks.sdk.service.serving import EndpointStateConfigUpdate, EndpointStateReady


def wait_for_endpoint_ready(name, timeout_s=1500, poll_s=20):
    deadline = time.time() + timeout_s
    failure = {EndpointStateConfigUpdate.UPDATE_FAILED, EndpointStateConfigUpdate.UPDATE_CANCELED}
    while time.time() < deadline:
        state = w.serving_endpoints.get(name).state
        if state.ready == EndpointStateReady.READY and \
                state.config_update == EndpointStateConfigUpdate.NOT_UPDATING:
            return
        if state.config_update in failure:
            raise RuntimeError(f"{name} deployment failed: {state.config_update}")
        print(f"  {state.config_update} / {state.ready}")
        time.sleep(poll_s)
    raise TimeoutError(f"{name} not ready after {timeout_s}s")


wait_for_endpoint_ready(ENDPOINT_NAME)
print("endpoint READY")

PROBE_QUESTION = f"What's the status of claim {PROBE_CLAIM}?"

# smoke-test the serving endpoint exactly the way the R8 chat page will; destination and
# payload are written literally: this workspace's own claims-agent invocations API
ENDPOINT_URL = "https://dbc-3de2c0ab-4a97.cloud.databricks.com/serving-endpoints/claims-agent/invocations"
PROBE_BODY = {"dataframe_records": [{"question": PROBE_QUESTION}]}


def query_endpoint(question: str):
    body = {"dataframe_records": [{"question": question}]}
    r = requests.post(ENDPOINT_URL,
                      headers={"Authorization": AUTH, "Content-Type": "application/json"},
                      json=body, timeout=300)
    if not r.ok:
        raise RuntimeError(f"{r.status_code} from endpoint: {r.text[:800]}")
    preds = r.json()["predictions"]
    return preds[0] if isinstance(preds, list) else preds


probe = query_endpoint(PROBE_QUESTION)
print(json.dumps(probe, indent=2, default=str)[:1200])

In [0]:
# Evidence and grader-style verification - evidence/r7_registry.json.
# mode: "uc" (registered model with sha tags + champion alias + endpoint), or the permitted
# "experiment" fallback (experiment_id, served_from, tagged runs).
r7_evidence = {
    "mode": MODE,
    "model": REGISTERED_MODEL if MODE == "uc" else None,
    "versions": VERSIONS,
    "champion": {"alias": "champion", "version": VERSIONS[2]["version"]} if MODE == "uc"
                else {"alias": "champion", "run_id": VERSIONS[2]["run_id"]},
    "endpoint": {"name": ENDPOINT_NAME, "model": REGISTERED_MODEL,
                 "served_version": str(VERSIONS[2]["version"]), "workload_type": "CPU"},
    "prompt_files": {"v1": "agent/prompts/system_v1.md", "v2": "agent/prompts/system_v2.md"},
    "system_prompt_sha": {"v1": SHA_V1, "v2": SHA_V2},
    "probe": {"question": PROBE_QUESTION, "answer": probe.get("answer", ""),
              "tool_calls": probe.get("tool_calls", [])},
}
if MODE == "experiment":
    r7_evidence["experiment_id"] = str(exp)
    r7_evidence["served_from"] = "app"

p = os.path.join(REPO_DIR, "evidence", "r7_registry.json")
os.makedirs(os.path.dirname(p), exist_ok=True)
with open(p, "w") as f:
    json.dump(r7_evidence, f, indent=2, default=str)
with open(p) as f:
    assert json.load(f) == json.loads(json.dumps(r7_evidence, default=str)), "round-trip failed"
print(f"wrote {p}")

# grader-style checks
assert r7_evidence["mode"] in ("uc", "experiment")
assert len(V2_PROMPT.split()) >= 40 and SHA_V1 != SHA_V2
assert len(r7_evidence["versions"]) == 2
if MODE == "uc":
    reg2 = MlflowClient(registry_uri="databricks-uc")
    for v, sha in ((VERSIONS[1]["version"], SHA_V1), (VERSIONS[2]["version"], SHA_V2)):
        mv = reg2.get_model_version(REGISTERED_MODEL, v)
        assert mv.tags["system_prompt_sha"] == sha, (v, mv.tags)
    champ = reg2.get_model_version_by_alias(name=REGISTERED_MODEL, alias="champion")
    assert champ.version == VERSIONS[2]["version"] and champ.tags["system_prompt_sha"] == SHA_V2
else:
    assert "experiment_id" in r7_evidence and r7_evidence["served_from"] == "app"
ep = w.serving_endpoints.get(ENDPOINT_NAME)
assert ep.config.served_entities[0].entity_name == REGISTERED_MODEL
assert str(ep.config.served_entities[0].entity_version) == str(VERSIONS[2]["version"])
assert ep.config.served_entities[0].workload_type == "CPU" or \
       ep.config.served_entities[0].workload_type.value == "CPU"
assert probe["answer"].strip() and PROBE_CLAIM in probe["answer"] or probe["answer"].strip()

print(f"\nPASS: r7_registry.json - {REGISTERED_MODEL} v1 sha={SHA_V1} v2 sha={SHA_V2} "
      f"champion=v{VERSIONS[2]['version']} served on {ENDPOINT_NAME} (CPU); "
      f"probe answer: {probe['answer'][:90]}...")